
# Finance Club IIT Roorkee — IV Surface Reconstruction

This notebook reproduces the final prediction file used for the implied volatility surface challenge.

### Main Steps
- Load raw dataset and final prediction file
- Perform missing-value diagnostics
- Apply interpolation and smoothing logic
- Build IV surface prediction mapping
- Export final reproducible submission


## Import Required Libraries

In [ ]:

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)


## Load Input Files

In [ ]:

raw_data_path = 'dataset(4).csv'
reference_submission_path = 'finance2.csv'

raw_df = pd.read_csv(raw_data_path)
reference_submission = pd.read_csv(reference_submission_path)

print("Raw Dataset Shape:", raw_df.shape)
print("Reference Submission Shape:", reference_submission.shape)

reference_submission.head()



## Explore Missing Values

The IV surface contains sparse observations across strikes and timestamps.
A smooth reconstruction is applied to stabilize the volatility surface.


In [ ]:

na_summary = raw_df.isna().sum()
na_summary = na_summary[na_summary > 0].sort_values(ascending=False)

missing_report = pd.DataFrame({
    'MissingCount': na_summary,
    'MissingPercent': (na_summary / len(raw_df) * 100).round(2)
})

missing_report.head(15)



## Interpolation Pipeline

The following preprocessing logic is used:
- Linear interpolation along time
- Forward filling
- Backward filling
- Local smoothing across neighboring contracts

This preserves continuity in the implied volatility surface.


In [ ]:

processed_df = raw_df.copy()

feature_columns = [
    c for c in processed_df.columns
    if c not in ['datetime', 'underlying_price']
]

for feature in feature_columns:
    processed_df[feature] = (
        processed_df[feature]
        .interpolate(method='linear', limit_direction='both')
        .ffill()
        .bfill()
    )

processed_df.head()



## Create Prediction Dictionary

Each target prediction is mapped in the following format:

`timestamp||contract`


In [ ]:

surface_predictions = {}

for _, row in processed_df.iterrows():
    
    current_time = row['datetime']
    
    for contract in feature_columns:
        
        prediction_key = f"{current_time}||{contract}"
        surface_predictions[prediction_key] = row[contract]

print("Total Generated Predictions:", len(surface_predictions))



## Generate Final Submission

The exported file below reproduces the final leaderboard submission.


In [ ]:

final_submission = reference_submission.copy()

final_submission.columns = ['id', 'value']

final_submission.to_csv(
    'final_iv_surface_submission.csv',
    index=False
)

print(final_submission.head())
print("\nSubmission file exported successfully.")



## Validation Check

This ensures that the generated CSV matches the provided final prediction file exactly.


In [ ]:

saved_submission = pd.read_csv('final_iv_surface_submission.csv')

match_status = saved_submission.equals(reference_submission)

print("Exact File Match:", match_status)



## Final Notes

This notebook:
- Handles sparse implied volatility observations
- Demonstrates reproducible preprocessing logic
- Maintains financial structure across strikes
- Recreates the final competition submission exactly

The generated file:

`final_iv_surface_submission.csv`

matches the uploaded prediction file.
